In [2]:
!pip install -q -U datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

In [4]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("hugging-face-read-dataset")
login(token=hf_token)

In [5]:
dataset = load_dataset(
    "MChoopani/Persian-event-reminder",
    token=hf_token
)

persian-event-reminder.csv:   0%|          | 0.00/97.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
print(dataset)
print(dataset["train"][0])
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['ID', 'text', 'intent'],
        num_rows: 949
    })
})
{'ID': 1, 'text': 'فردا ساعت ۹ قرار دارم\u200cقبلش بهم\u200cبگو', 'intent': 'EVENT'}
['ID', 'text', 'intent']


In [7]:
from collections import Counter

intent_counts = Counter(dataset["train"]["intent"])

for intent, count in intent_counts.most_common():
    print(f"{intent}: {count}")

RECURRING_EVENT_REMINDER: 322
RECURRING_EVENT: 318
EVENT: 309


In [8]:
import pandas as pd

df = dataset["train"].to_pandas()

display(
    df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

,intent,count
0,RECURRING_EVENT_REMINDER,322
1,RECURRING_EVENT,318
2,EVENT,309


In [9]:
print("تعداد کل:", len(df))
print("متن‌های تکراری:", df["text"].duplicated().sum())
print("متن و Intent تکراری:", df.duplicated(subset=["text", "intent"]).sum())

تعداد کل: 949
متن‌های تکراری: 0
متن و Intent تکراری: 0


In [10]:
conflicting_texts = (
    df.groupby("text")["intent"]
    .nunique()
    .loc[lambda x: x > 1]
)

print("تعداد متن‌های دارای Intent متناقض:", len(conflicting_texts))

display(
    df[df["text"].isin(conflicting_texts.index)]
    .sort_values("text")
)

تعداد متن‌های دارای Intent متناقض: 0


,ID,text,intent


In [11]:
import re

def normalize_persian_text(text):
    text = str(text)

    # یکسان‌سازی حروف عربی و فارسی
    text = text.replace("ي", "ی")
    text = text.replace("ى", "ی")
    text = text.replace("ك", "ک")

    # حذف کاراکترهای کنترلی نامرئی
    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")
    text = text.replace("\ufeff", "")

    # تبدیل Tab، Enter و چند فاصله به یک فاصله
    text = re.sub(r"\s+", " ", text)

    return text.strip()

dataset = dataset.map(
    lambda example: {
        "text": normalize_persian_text(example["text"])
    }
)

Map:   0%|          | 0/949 [00:00<?, ? examples/s]

In [12]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['ID', 'text', 'intent'],
        num_rows: 949
    })
})


In [13]:
print(dataset["train"])

Dataset({
    features: ['ID', 'text', 'intent'],
    num_rows: 949
})


In [14]:
full_dataset = dataset["train"]

# کپی intent در ستون label
full_dataset = full_dataset.map(
    lambda row: {"label": row["intent"]}
)

# تبدیل label از متن به ClassLabel عددی
full_dataset = full_dataset.class_encode_column("label")

print(full_dataset.features)

Map:   0%|          | 0/949 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/949 [00:00<?, ? examples/s]

{'ID': Value('int64'), 'text': Value('string'), 'intent': Value('string'), 'label': ClassLabel(names=['EVENT', 'RECURRING_EVENT', 'RECURRING_EVENT_REMINDER'])}


In [15]:
print(full_dataset)

Dataset({
    features: ['ID', 'text', 'intent', 'label'],
    num_rows: 949
})


In [16]:
first_split = full_dataset.train_test_split(
    test_size=0.20,
    seed=42,
    stratify_by_column="label"
)

print(first_split)

DatasetDict({
    train: Dataset({
        features: ['ID', 'text', 'intent', 'label'],
        num_rows: 759
    })
    test: Dataset({
        features: ['ID', 'text', 'intent', 'label'],
        num_rows: 190
    })
})


In [17]:
print(first_split["train"].features)
print(first_split["train"][0])

{'ID': Value('int64'), 'text': Value('string'), 'intent': Value('string'), 'label': ClassLabel(names=['EVENT', 'RECURRING_EVENT', 'RECURRING_EVENT_REMINDER'])}
{'ID': 649, 'text': 'هر سه\u200cشنبه ساعت ۹ صبح دانشگاه میرم نیم ساعت قبلش بهم بگو', 'intent': 'RECURRING_EVENT_REMINDER', 'label': 2}


In [18]:
from datasets import DatasetDict

train_validation = first_split["train"].train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column="label"
)

final_dataset = DatasetDict({
    "train": train_validation["train"],
    "validation": train_validation["test"],
    "test": first_split["test"]
})

print(final_dataset)

DatasetDict({
    train: Dataset({
        features: ['ID', 'text', 'intent', 'label'],
        num_rows: 645
    })
    validation: Dataset({
        features: ['ID', 'text', 'intent', 'label'],
        num_rows: 114
    })
    test: Dataset({
        features: ['ID', 'text', 'intent', 'label'],
        num_rows: 190
    })
})


In [19]:
from collections import Counter

for split_name in final_dataset:
    print(
        split_name,
        Counter(final_dataset[split_name]["label"])
    )

train Counter({2: 219, 1: 216, 0: 210})
validation Counter({2: 39, 1: 38, 0: 37})
test Counter({1: 64, 2: 64, 0: 62})


In [20]:
from transformers import AutoTokenizer

MODEL_NAME = "HooshvareLab/distilbert-fa-zwnj-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/500 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/426k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

In [21]:
import numpy as np

token_lengths = [
    len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
    for text in final_dataset["train"]["text"]
]

print("Min:", np.min(token_lengths))
print("Mean:", np.mean(token_lengths))
print("Median:", np.median(token_lengths))
print("95th percentile:", np.percentile(token_lengths, 95))
print("99th percentile:", np.percentile(token_lengths, 99))
print("Max:", np.max(token_lengths))

Min: 6
Mean: 12.756589147286821
Median: 12.0
95th percentile: 19.0
99th percentile: 21.0
Max: 24


In [22]:
MAX_LENGTH = 48

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = final_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["ID", "text", "intent"]
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/645 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 645
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 114
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 190
    })
})
{'label': 0, 'input_ids': [2, 7089, 5213, 695, 3148, 4215, 4599, 2008, 2597, 19957, 10519, 20261, 4838, 595, 6432, 2223, 1112, 29171, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [23]:
tokenized_dataset = tokenized_dataset.remove_columns(
    "token_type_ids"
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 645
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 114
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 190
    })
})
{'label': 0, 'input_ids': [2, 7089, 5213, 695, 3148, 4215, 4599, 2008, 2597, 19957, 10519, 20261, 4838, 595, 6432, 2223, 1112, 29171, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [24]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8
)

In [25]:
id2label = {
    0: "EVENT",
    1: "RECURRING_EVENT",
    2: "RECURRING_EVENT_REMINDER"
}

label2id = {
    "EVENT": 0,
    "RECURRING_EVENT": 1,
    "RECURRING_EVENT_REMINDER": 2
}

In [26]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "HooshvareLab/distilbert-fa-zwnj-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  303MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: HooshvareLab/distilbert-fa-zwnj-base
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(42000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [28]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [29]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./intent_distilbert_fa",

    # ارزیابی و ذخیره در پایان هر Epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    # نگهداری بهترین مدل بر اساس Macro-F1
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    # Hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_steps=33,

    # Logging
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",

    # Reproducibility
    seed=42,
    data_seed=42,

    # فقط روی GPU فعال می‌شود
    fp16=torch.cuda.is_available(),
)

In [30]:
import math

BATCH_SIZE = 16
EPOCHS = 8

steps_per_epoch = math.ceil(
    len(tokenized_dataset["train"]) / BATCH_SIZE
)

total_training_steps = steps_per_epoch * EPOCHS

warmup_steps = round(
    total_training_steps * 0.10
)

print("Steps per epoch:", steps_per_epoch)
print("Total planned steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Steps per epoch: 41
Total planned steps: 328
Warmup steps: 33


In [31]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,
    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [32]:
print("Device:", model.device)
print("Train samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))
print("Test samples:", len(tokenized_dataset["test"]))
print("Warmup steps:", training_args.warmup_steps)

Device: cuda:0
Train samples: 645
Validation samples: 114
Test samples: 190
Warmup steps: 33


In [33]:
print("Trainer device:", trainer.args.device)
print("FP16 enabled:", trainer.args.fp16)

Trainer device: cuda:0
FP16 enabled: True


In [34]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.487211,0.203525,0.991228,0.991667,0.990991,0.991214
2,0.011785,0.008198,1.000000,1.000000,1.000000,1.000000
3,0.022514,0.002247,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.487211,0.203525,0.991228,0.991667,0.990991,0.991214
2,0.011785,0.008198,1.000000,1.000000,1.000000,1.000000
3,0.022514,0.002247,1.000000,1.000000,1.000000,1.000000
4,0.002904,0.001530,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation F1:", trainer.state.best_metric)
print("Final global step:", trainer.state.global_step)
print("Best epoch:", trainer.state.best_model_checkpoint)

Best checkpoint: ./intent_distilbert_fa/checkpoint-82
Best validation F1: 1.0
Final global step: 164
Best epoch: ./intent_distilbert_fa/checkpoint-82


In [36]:
print(train_result)

TrainOutput(global_step=164, training_loss=0.22816357782065141, metrics={'train_runtime': 55.2335, 'train_samples_per_second': 93.422, 'train_steps_per_second': 5.938, 'total_flos': 15844625322480.0, 'train_loss': 0.22816357782065141, 'epoch': 4.0})


In [37]:
validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

validation_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.002904,0.008198,4,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.008198019117116928,
 'eval_accuracy': 1.0,
 'eval_precision_macro': 1.0,
 'eval_recall_macro': 1.0,
 'eval_f1_macro': 1.0}

In [38]:
test_results = trainer.evaluate(
    tokenized_dataset["test"],
    metric_key_prefix="test"
)

test_results

[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.002904,0.021012,4,0.994737,0.994709,0.994792,0.994709


{'test_loss': 0.021012134850025177,
 'test_accuracy': 0.9947368421052631,
 'test_precision_macro': 0.9947089947089948,
 'test_recall_macro': 0.9947916666666666,
 'test_f1_macro': 0.9947086614173228}

In [39]:
import numpy as np

test_output = trainer.predict(
    tokenized_dataset["test"]
)

y_true = test_output.label_ids
y_pred = np.argmax(
    test_output.predictions,
    axis=-1
)

wrong_indices = np.where(y_true != y_pred)[0]

print("Number of errors:", len(wrong_indices))

label_names = (
    final_dataset["train"]
    .features["label"]
    .names
)

for index in wrong_indices:
    row = final_dataset["test"][int(index)]

    print("-" * 80)
    print("ID:", row["ID"])
    print("Text:", row["text"])
    print("True:", label_names[int(y_true[index])])
    print("Predicted:", label_names[int(y_pred[index])])

Number of errors: 1
--------------------------------------------------------------------------------
ID: 410
Text: هفته ای یک بار یادم بنداز
True: RECURRING_EVENT
Predicted: EVENT


In [40]:
SAVE_PATH = "./intent_event_reminder_distilbert_fa_best"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./intent_event_reminder_distilbert_fa_best/tokenizer_config.json',
 './intent_event_reminder_distilbert_fa_best/tokenizer.json')

In [41]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [42]:
SAVE_PATH = "/content/drive/MyDrive/models/./intent_event_reminder_distilbert_fa_best"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/models/./intent_event_reminder_distilbert_fa_best/tokenizer_config.json',
 '/content/drive/MyDrive/models/./intent_event_reminder_distilbert_fa_best/tokenizer.json')

In [43]:
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/models/./intent_event_reminder_distilbert_fa_best/tokenizer_config.json',
 '/content/drive/MyDrive/models/./intent_event_reminder_distilbert_fa_best/tokenizer.json')

In [44]:
import os

for filename in os.listdir(SAVE_PATH):
    print(filename)

config.json
model.safetensors
tokenizer_config.json
tokenizer.json
training_args.bin


In [45]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

SAVE_PATH = (
    "/content/drive/MyDrive/models/"
    "intent_event_reminder_distilbert_fa_best"
)

loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_PATH
)

print("Tokenizer loaded:", type(loaded_tokenizer).__name__)
print("Model loaded:", type(loaded_model).__name__)
print("Labels:", loaded_model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokenizer loaded: BertTokenizer
Model loaded: DistilBertForSequenceClassification
Labels: {0: 'EVENT', 1: 'RECURRING_EVENT', 2: 'RECURRING_EVENT_REMINDER'}


In [57]:
import torch

text = "هر هفته صبح کلاس زبان دارم یک ساعت قبلش یادم بنداز ه"

inputs = loaded_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=48
)

# در صورت تولید token_type_ids، چون DistilBERT نیاز ندارد حذف می‌کنیم
inputs.pop("token_type_ids", None)

loaded_model.eval()

with torch.no_grad():
    logits = loaded_model(**inputs).logits

probabilities = torch.softmax(logits, dim=-1)[0]
predicted_id = int(probabilities.argmax())

print("Predicted ID:", predicted_id)
print("Intent:", loaded_model.config.id2label[predicted_id])
print("Confidence:", float(probabilities[predicted_id]))

Predicted ID: 2
Intent: RECURRING_EVENT_REMINDER
Confidence: 0.984512984752655
